#### Taking extracted neuroimaging data and adding it to Scott 10K
#### 1. Add parcellated data from brain2cog
#### 2. Add b2c data after normalisation
#### 3. Add recon-all data

In [ ]:
%%bash
rm -v /rds/general/project/c3nl_scott_students/live/data/sankeith/scott_10k_reconall_scraped/*.csv
mkdir -p /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/add_neuroimaging_data
mkdir -p /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/add_neuroimaging_data

### Add recon-all data - In particular:

#### ASEG stats
#### Destrieux volume
#### Desikan volume
#### Desikan thickness
#### Destrieux thickness

In [1]:
import os
import pandas as pd

reconall_files = os.listdir('/rds/general/project/c3nl_scott_students/live/data/sankeith/scott_10k_reconall_scraped')
path = '/rds/general/project/c3nl_scott_students/live/data/sankeith/scott_10k_reconall_scraped'
for file in reconall_files:
    if file.endswith('.txt'):
        try:
            ra_path = os.path.join(path,file)
            ra_basename = os.path.basename(ra_path).strip('.txt')
            print(f'Basename: {ra_basename}')
            ra_df = pd.read_csv(ra_path, delimiter = '\t', low_memory = False)
            old_name = ra_df.columns.tolist()[0]
            print(old_name, ra_basename)
            ra_df.rename(columns = {old_name: 'DATA_KEY'}, inplace = True)
            ra_df['DATA_KEY'] = ra_df['DATA_KEY'].apply(lambda x: x.strip('/'))
    
    
            old_col_names = ra_df.columns.tolist()[1:]
            
            def updatecolname(name):
                return name + '_' + ra_basename
    
            new_col_names = [updatecolname(col) for col in old_col_names]

            print(new_col_names)
    
            assert len(old_col_names) == len(new_col_names)
    
            rename_dict = dict(zip(old_col_names, new_col_names))
    
            ra_df.rename(columns = rename_dict, inplace = True)
                
            print(f'New column names: {ra_df.columns.tolist()[-1:]}')
            
            new_path = f'/rds/general/project/c3nl_scott_students/live/data/sankeith/scott_10k_reconall_scraped/{ra_basename}.csv'
            print(new_path)
            ra_df.to_csv(new_path, index = False)
            
        except IsADirectoryError:
            continue


Basename: rh.BA_exvivo.area
rh.BA_exvivo.area rh.BA_exvivo.area
['rh_BA1_exvivo_area_rh.BA_exvivo.area', 'rh_BA2_exvivo_area_rh.BA_exvivo.area', 'rh_BA3a_exvivo_area_rh.BA_exvivo.area', 'rh_BA3b_exvivo_area_rh.BA_exvivo.area', 'rh_BA4a_exvivo_area_rh.BA_exvivo.area', 'rh_BA4p_exvivo_area_rh.BA_exvivo.area', 'rh_BA6_exvivo_area_rh.BA_exvivo.area', 'rh_BA44_exvivo_area_rh.BA_exvivo.area', 'rh_BA45_exvivo_area_rh.BA_exvivo.area', 'rh_V1_exvivo_area_rh.BA_exvivo.area', 'rh_V2_exvivo_area_rh.BA_exvivo.area', 'rh_MT_exvivo_area_rh.BA_exvivo.area', 'rh_perirhinal_exvivo_area_rh.BA_exvivo.area', 'rh_entorhinal_exvivo_area_rh.BA_exvivo.area', 'rh_WhiteSurfArea_area_rh.BA_exvivo.area', 'BrainSegVolNotVent_rh.BA_exvivo.area', 'eTIV_rh.BA_exvivo.area']
New column names: ['eTIV_rh.BA_exvivo.area']
/rds/general/project/c3nl_scott_students/live/data/sankeith/scott_10k_reconall_scraped/rh.BA_exvivo.area.csv
Basename: aparc_thickness_rh
rh.aparc.thickness aparc_thickness_rh
['rh_bankssts_thickness_apar

### (Where appropriate) Normalise with eTIV/TIV.

### We should only normalise:

#### Desikan volume and thickness
#### Destrieux volume and thickness
#### ASEG volume

In [2]:
import pandas as pd, tqdm, os, numpy as np
from tqdm import tqdm

ra_path = '/rds/general/project/c3nl_scott_students/live/data/sankeith/scott_10k_reconall_scraped'
eTIV_cols = []
files_of_interest = ['aparc_thickness_lh.csv','aparc_thickness_rh.csv', # Desikan thickness
                    'lh.a2009s.thickness.csv', 'rh.a2009s.thickness.csv', # Destrieux thickness
                     'aparc_volume_lh.csv', 'aparc_volume_lh.csv', # Desikan volume
                     'lh.a2009s.volume.csv', 'rh.a2009s.volume.csv', # Destrieux volume
                    'aseg_stats.csv'] # ASEG volume
for file in files_of_interest:
    ra_df = pd.read_csv(os.path.join(ra_path,file))
    ra_df_cols = ra_df.columns.tolist()
    print(ra_df_cols)
    # Makes sure that eTIV is calculated identically in all CSVs
    for col in ra_df_cols:
        col_parts = col.split('_')
        if 'eTIV' in col_parts or 'EstimatedTotalIntraCranialVol' in col_parts:
            print(f"{file} has an eTIV column!") 
            eTIV_cols.append(ra_df[col].values)
print(len(eTIV_cols))

for ele in eTIV_cols:
    print(len(ele), end = ' ') 

## QC check - eTIV calculation isn't consistent across all datasets
i = 1
while i < len(eTIV_cols):
    try:
        assert np.array_equal(eTIV_cols[i], eTIV_cols[i-1]) == True
        i += 1
    except AssertionError:
        print(f"\neTIVs aren't equal - eTIV_cols[{i}] and eTIV_cols[{i-1}]")
        i+=1
        continue
                    
                    

['DATA_KEY', 'lh_bankssts_thickness_aparc_thickness_lh', 'lh_caudalanteriorcingulate_thickness_aparc_thickness_lh', 'lh_caudalmiddlefrontal_thickness_aparc_thickness_lh', 'lh_cuneus_thickness_aparc_thickness_lh', 'lh_entorhinal_thickness_aparc_thickness_lh', 'lh_fusiform_thickness_aparc_thickness_lh', 'lh_inferiorparietal_thickness_aparc_thickness_lh', 'lh_inferiortemporal_thickness_aparc_thickness_lh', 'lh_isthmuscingulate_thickness_aparc_thickness_lh', 'lh_lateraloccipital_thickness_aparc_thickness_lh', 'lh_lateralorbitofrontal_thickness_aparc_thickness_lh', 'lh_lingual_thickness_aparc_thickness_lh', 'lh_medialorbitofrontal_thickness_aparc_thickness_lh', 'lh_middletemporal_thickness_aparc_thickness_lh', 'lh_parahippocampal_thickness_aparc_thickness_lh', 'lh_paracentral_thickness_aparc_thickness_lh', 'lh_parsopercularis_thickness_aparc_thickness_lh', 'lh_parsorbitalis_thickness_aparc_thickness_lh', 'lh_parstriangularis_thickness_aparc_thickness_lh', 'lh_pericalcarine_thickness_aparc_t

In [3]:
import pandas as pd, numpy as np, os, tqdm
from tqdm import tqdm

diagnoses = pd.read_csv('/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/no_neuroimaging_data_scott10k_alliedhealth.csv', low_memory = False)[['DATA_KEY','DIAGNOSIS']]


ra_path = '/rds/general/project/c3nl_scott_students/live/data/sankeith/scott_10k_reconall_scraped'
files_of_interest = ['aparc_thickness_lh.csv','aparc_thickness_rh.csv', # Desikan thickness
                    'lh.a2009s.thickness.csv', 'rh.a2009s.thickness.csv', # Destrieux thickness
                     'aparc_volume_lh.csv', 'aparc_volume_rh.csv', # Desikan volume
                     'lh.a2009s.volume.csv', 'rh.a2009s.volume.csv', # Destrieux volume
                    'aseg_stats.csv'] # ASEG volume

for file in files_of_interest:
    if file not in ['aparc_thickness_lh.csv','aparc_thickness_rh.csv', 'lh.a2009s.thickness.csv', 'rh.a2009s.thickness.csv']:
        ra_df = pd.read_csv(os.path.join(ra_path,file))
        ra_df_cols = ra_df.columns.tolist()
        for col in ra_df_cols:
            col_parts = col.split('_')
            # Normalisation with eTIV
            if 'eTIV' in col_parts or 'EstimatedTotalIntraCranialVol' in col_parts: # NB: wmparc uses 'EstimatedTotalIntracranialVolume' so it gets missed: I'll deal with this later
                print('Starting eTIV normalisation...')
                etiv_col_name = col
                print(etiv_col_name)
                
                data_key = ra_df['DATA_KEY']
                df_to_normalise = ra_df.drop(columns = ['DATA_KEY', etiv_col_name])

                normalised_df = df_to_normalise.div(ra_df[etiv_col_name], axis = 0) #NB: normalised df will NOT have eTIV values, since they will all just be 1

                norm_df = pd.concat([data_key ,normalised_df], axis = 1)
                assert data_key.index.equals(norm_df.index)
                norm_df_with_diags = pd.merge(diagnoses, norm_df, how = 'left', on = 'DATA_KEY')
                print(norm_df_with_diags.shape)
                norm_df_with_diags.to_csv(f'/rds/general/project/c3nl_scott_students/live/data/sankeith/scott_10k_reconall_scraped/norm_{file}', index = False)
    else:
        print(f'{file} is cortical thickness values - skipping ICV normalisation')
        thickness_ra_path = os.path.join(ra_path,file)
        norm_df_with_diags = pd.merge(diagnoses, pd.read_csv(thickness_ra_path, low_memory = False), how = 'inner', on = 'DATA_KEY')
        print(norm_df_with_diags.shape)
        for col in norm_df_with_diags.columns.tolist():
            col_parts = col.split('_')
            if 'eTIV' in col_parts or 'EstimatedTotalIntraCranialVol' in col_parts: # NB: wmparc uses 'EstimatedTotalIntracranialVolume' so it gets missed: I'll deal with this later
                etiv_col_name = col
                print(f'eTIV column detected: {etiv_col_name}. Removing...')
                norm_df_with_diags.drop(columns = [etiv_col_name])
                norm_df_with_diags.to_csv(f'/rds/general/project/c3nl_scott_students/live/data/sankeith/scott_10k_reconall_scraped/norm_{file}', index = False)

aparc_thickness_lh.csv is cortical thickness values - skipping ICV normalisation
(15685, 39)
eTIV column detected: eTIV_aparc_thickness_lh. Removing...
aparc_thickness_rh.csv is cortical thickness values - skipping ICV normalisation
(15685, 39)
eTIV column detected: eTIV_aparc_thickness_rh. Removing...
lh.a2009s.thickness.csv is cortical thickness values - skipping ICV normalisation
(15685, 79)
eTIV column detected: eTIV_lh.a2009s.thickness. Removing...
rh.a2009s.thickness.csv is cortical thickness values - skipping ICV normalisation
(15685, 79)
eTIV column detected: eTIV_rh.a2009s.thickness. Removing...
Starting eTIV normalisation...
eTIV_aparc_volume_lh
(15733, 37)
Starting eTIV normalisation...
eTIV_aparc_volume_rh
(15733, 37)
Starting eTIV normalisation...
eTIV_lh.a2009s.volume
(15733, 77)
Starting eTIV normalisation...
eTIV_rh.a2009s.volume
(15733, 77)
Starting eTIV normalisation...
EstimatedTotalIntraCranialVol_aseg_stats
(15733, 65)


In [4]:
import pandas as pd, numpy as np, os, tqdm
from tqdm import tqdm

ra_path = '/rds/general/project/c3nl_scott_students/live/data/sankeith/scott_10k_reconall_scraped'
files_of_interest = ['aparc_thickness_lh.csv','aparc_thickness_rh.csv', # Desikan thickness
                    'lh.a2009s.thickness.csv', 'rh.a2009s.thickness.csv', # Destrieux thickness
                     'aparc_volume_lh.csv', 'aparc_volume_rh.csv', # Desikan volume
                     'lh.a2009s.volume.csv', 'rh.a2009s.volume.csv', # Destrieux volume
                    'aseg_stats.csv'] # ASEG volume

for file in files_of_interest:
    norm_ra_df = pd.read_csv(os.path.join(ra_path, f'norm_{file}'))

    # Extract CN values #

    cn_norm_ra_df = norm_ra_df[norm_ra_df['DIAGNOSIS'] == 1]
    cn_norm_ra_df.drop(columns = ['DATA_KEY', 'DIAGNOSIS'], inplace = True)
    # Calculate CN mean and std #

    cn_norm_ra_df_mean = cn_norm_ra_df.mean()
    cn_norm_ra_df_std = cn_norm_ra_df.std()
    
    # z-score! #

    norm_ra_df_datakey_n_diags = norm_ra_df[['DATA_KEY', 'DIAGNOSIS']]
    norm_ra_df.drop(columns = ['DATA_KEY', 'DIAGNOSIS'], inplace = True)
    zscored_values = (norm_ra_df - cn_norm_ra_df_mean) / cn_norm_ra_df_std

    assert norm_ra_df_datakey_n_diags.index.equals(zscored_values.index)

    zscored_df = pd.concat([norm_ra_df_datakey_n_diags, zscored_values], axis=1)
    print(zscored_df.shape)
    zscored_df.to_csv(f'/rds/general/project/c3nl_scott_students/live/data/sankeith/scott_10k_reconall_scraped/z_norm_{file}', index = False)



    

(15685, 39)
(15685, 39)
(15685, 79)
(15685, 79)
(15733, 37)
(15733, 37)
(15733, 77)
(15733, 77)
(15733, 65)


### Add to scott_10k...

1. Desikan VBM ROIs
2. Desikan coeffs
3. Desikan regression stats
4. Desikan flipped coeffs
5. Desikan  flipped regression stats


1. Destrieux VBM ROIs
2. Destrieux coeffs
3. Destrieux regression stats
4. Destrieux flipped coeffs
5. Destrieux flipped regression stats


1. Aseg stats (volume)
2. Desikan volume (lh, rh)
3. Destrieux volume (lh, rh)
4. Desikan thickness (lh, rh)
5. Destrieux volume (lh, rh)

In [1]:
import pandas as pd, pandasql as ps, os, re

df = pd.read_csv('/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/no_neuroimaging_data_scott10k_alliedhealth.csv', low_memory = False)

#Add VBM files (files 1-10)
path = '/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/'
selected_files = ['gm_only_scraped_desikan_stats', 'desikan_coeffs', 'desikan_stats', 'flipped_desikan_coeffs', 'flipped_desikan_stats',
                 'gm_only_scraped_destrieux_stats', 'destrieux_coeffs', 'destrieux_stats', 'flipped_destrieux_coeffs', 'flipped_destrieux_stats']

for file in selected_files:
        vbm_path = os.path.join(path, f'{file}.csv')
        vbm_df = pd.read_csv(vbm_path, low_memory = False)
        file_name_parts  = file.split('_')
        if file in ['desikan_stats', 'destrieux_stats', 'flipped_desikan_stats', 'flipped_destrieux_stats']:
            vbm_df = vbm_df.loc[:, ~vbm_df.columns.str.contains('^Unnamed')]           
            col_prefix = '_'.join(file_name_parts[:-1]) + '_'
            excluded = ['DATA_KEY']
            vbm_df.rename(columns = lambda x: col_prefix + x if x not in excluded else x, inplace = True)
        elif file in ['gm_only_scraped_desikan_stats']:
            vbm_df.drop(columns=['Unnamed: 0'], errors='ignore', inplace=True)            
            col_prefix = 'desikan_vbm_'
            excluded = ['DATA_KEY']
            vbm_df.rename(columns = lambda x: col_prefix + x if x not in excluded else x, inplace = True)
        elif file in ['gm_only_scraped_destrieux_stats']:
            vbm_df = vbm_df.loc[:, ~vbm_df.columns.str.contains('^Unnamed')]           
            col_prefix = 'destrieux_vbm_'
            excluded = ['DATA_KEY']
            vbm_df.rename(columns = lambda x: col_prefix + x if x not in excluded else x, inplace = True)
        df = pd.merge(df, vbm_df, how = 'left', on = 'DATA_KEY')
        print(df.shape, end = ' ')
df.to_csv('/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/w_vbm_scott10k_alliedhealth.csv', index = False)    
print(df.columns.tolist())

(15733, 163) (15733, 182) (15733, 185) (15733, 204) (15733, 207) (15733, 373) (15733, 392) (15733, 395) (15733, 414) (15733, 417) ['PTID', 'RID', 'SITE_ID', 'DATA_KEY', 'T1_YEAR', 'T1_MON', 'T1_DAY', 'VISDATE', 'EXAMDATE_4WKS_LATER', 'EXAMDATE_4WKS_B4', 'T1_PATH', 'MWC1T1_PATH', 'PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTAGE', 'GENOTYPE', 'EXAMDATE_MRIFLDSTRNGTH', 'FIELD_STRENGTH', 'DATEDIFFS_MRIFLDSTRNGTH', 'EXAMDATE_DXSUM', 'PHASE', 'DIAGNOSIS', 'DATEDIFFS_DXSUM', 'EXAMDATE_MMSE', 'MMSCORE', 'DATEDIFFS_MMSE', 'EXAMDATE_MOCA', 'MOCA', 'DATEDIFFS_MOCA', 'EXAMDATE_NEUROBAT', 'LIMMTOTAL', 'CLOCKSCOR', 'LDELTOTAL', 'LDELCUE', 'ANART', 'DATEDIFFS_NEUROBAT', 'EXAMDATE_CDR', 'CDGLOBAL', 'DATEDIFFS_CDR', 'EXAMDATE_FAQ', 'FAQTOTAL', 'DATEDIFFS_FAQ', 'EXAMDATE_NPIQ', 'NPISCORE', 'DATEDIFFS_NPIQ', 'EXAMDATE_GDSCALE', 'GDTOTAL', 'DATEDIFFS_GDSCALE', 'EXAMDATE_MODHACH', 'HMSCORE', 'DATEDIFFS_MODHACH', 'EXAMDATE_ADAS13', 'TOTAL13', 'DATEDIFFS_ADAS13', 'EXAMDATE_P217', 'P217_DILUTION_CORRECTED_CONC', 'P217

In [2]:
import pandas as pd, pandasql as ps, os

df = pd.read_csv('/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/w_vbm_scott10k_alliedhealth.csv', low_memory = False)    
print(df.shape)
path = '/rds/general/project/c3nl_scott_students/live/data/sankeith/scott_10k_reconall_scraped'
selected_files = ['aparc_thickness_lh.csv','aparc_thickness_rh.csv', # Desikan thickness
                    'lh.a2009s.thickness.csv', 'rh.a2009s.thickness.csv', # Destrieux thickness
                     'aparc_volume_lh.csv', 'aparc_volume_rh.csv', # Desikan volume
                     'lh.a2009s.volume.csv', 'rh.a2009s.volume.csv', # Destrieux volume
                    'aseg_stats.csv'] # ASEG volume
        

for file in selected_files:
    try:
        sbm_path = os.path.join(path,f'z_norm_{file}')
        if os.path.exists(sbm_path):
            sbm_df = pd.read_csv(sbm_path, low_memory = False)
        df = pd.merge(df, sbm_df, how = 'left', on = ['DATA_KEY', 'DIAGNOSIS'], suffixes = (None, None))
        print(df.shape)
    except Exception:
        raise Exception
df.to_csv('/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/sbm_w_vbm_scott10k_alliedhealth.csv', index = False)  
print(df.columns.tolist())

(15733, 417)
(15733, 454)
(15733, 491)
(15733, 568)
(15733, 645)
(15733, 680)
(15733, 715)
(15733, 790)
(15733, 865)
(15733, 928)
['PTID', 'RID', 'SITE_ID', 'DATA_KEY', 'T1_YEAR', 'T1_MON', 'T1_DAY', 'VISDATE', 'EXAMDATE_4WKS_LATER', 'EXAMDATE_4WKS_B4', 'T1_PATH', 'MWC1T1_PATH', 'PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTAGE', 'GENOTYPE', 'EXAMDATE_MRIFLDSTRNGTH', 'FIELD_STRENGTH', 'DATEDIFFS_MRIFLDSTRNGTH', 'EXAMDATE_DXSUM', 'PHASE', 'DIAGNOSIS', 'DATEDIFFS_DXSUM', 'EXAMDATE_MMSE', 'MMSCORE', 'DATEDIFFS_MMSE', 'EXAMDATE_MOCA', 'MOCA', 'DATEDIFFS_MOCA', 'EXAMDATE_NEUROBAT', 'LIMMTOTAL', 'CLOCKSCOR', 'LDELTOTAL', 'LDELCUE', 'ANART', 'DATEDIFFS_NEUROBAT', 'EXAMDATE_CDR', 'CDGLOBAL', 'DATEDIFFS_CDR', 'EXAMDATE_FAQ', 'FAQTOTAL', 'DATEDIFFS_FAQ', 'EXAMDATE_NPIQ', 'NPISCORE', 'DATEDIFFS_NPIQ', 'EXAMDATE_GDSCALE', 'GDTOTAL', 'DATEDIFFS_GDSCALE', 'EXAMDATE_MODHACH', 'HMSCORE', 'DATEDIFFS_MODHACH', 'EXAMDATE_ADAS13', 'TOTAL13', 'DATEDIFFS_ADAS13', 'EXAMDATE_P217', 'P217_DILUTION_CORRECTED_CONC', 'P217